# cc-finance-1.1 — Stage 1 EM (thought generation)

Snorkel-finance, Qwen3-4B. Two EM runs — **policy-scored** and **base-scored** — each 58 batches (2 epochs over 116 train tasks). E-step samples G=4 thoughts per logged action and scores them by the length-penalized action likelihood `p(x|s,z) − 0.15·(|z|/96)`; M-step is REINFORCE on (thought+action) with group-normalized EM weights.

This notebook plots the EM reward trajectory and marks the eval `best_step` (the `step_best` checkpoint we relabel from). The `_last` checkpoint (batch 58) is the other relabel source.

In [ ]:
import cc_finance_lib as L
em = L.load_em_runs()
for s, rows in em.items():
    xs, ys = L._series(rows, 'final_reward')
    bs = [r[k] for r in rows for k in r if k.endswith('final_reward_best_step')]
    print(f'{s:6s}: batches={len(xs):2d}  last_reward={ys[-1]:.3f}  eval_best_step={bs[-1] if bs else "n/a"}')

In [ ]:
L.plot_em(em, 'figs_finance/em_reward_curves.png');

## Reading it
- Dotted vertical = each scorer's eval `best_step` (what `step_best` captures); dashed = the 1-epoch boundary (batch 29).
- If `best_step` lands **past batch 29**, the 2nd epoch improved the thought policy (worth the extra compute); if it plateaus early (as airline did), a 1-epoch EM would suffice.
- Per-scorer `step_best` vs `step_last` both feed Stage 1.5 relabel → the `thoughts_policy`/`thoughts_base` (+ `_last`) SFT corpora compared in cc-finance-1.3.